

# Customer Segmentation & Lifetime Value (CLV) Prediction:

Note: This notebook is designed strictly for educational purposes to demonstrate the practical application of machine learning algorithms, specifically K-Means clustering and XGBoost regression, in a business context.

## 1. The Dataset:
This project analyzes real-world e-commerce transactional data to identify customer behavior patterns and predict future revenue.

[Source & Copyright Reference: Kaggle - Retail Customer Segmentation Analysis](https://www.kaggle.com/datasets/thedevastator/retail-customer-segmentation-analysis-using-cust)

## 2. Methodology & Pipeline:
The code executes an end-to-end data science pipeline divided into four distinct phases:

### Phase 1 | Data Cleaning:
Filtering out returns, cancellations, and missing Customer IDs to create a clean, mathematically sound baseline of 354,321 valid transactions.

### Phase 2 | RFM Engineering:
Grouping the raw data to calculate Recency (days since last purchase), Frequency (total unique invoices), and Monetary value (total historical spend) for each customer.

### Phase 3 | Behavioral Clustering:
Applying the K-Means algorithm to group customers into behavioral cohorts. The optimal number of clusters is verified mathematically using the Silhouette Score.

### Phase 4 | CLV Forecasting:
Splitting the timeline to prevent data leakage, and training an XGBoost Regressor to predict how much a customer will spend in the future based entirely on their historical RFM profile.

## 3. Key Results:

### Segmentation Findings:
The K-Means model identified 2 optimal clusters within the historical window:

* Low-Value Shoppers: 3,007 customers

* At-Risk / Sleepers: 28 high-variance customers

### Forecasting Accuracy:
The XGBoost model successfully captured a positive signal for future spending:

* R-squared (R²):0.3967 (explaining roughly 40% of the variance in future spending).

* Root Mean Squared Error (RMSE): $616.12

### Business Insights (Feature Importance):
When predicting future lifetime value, a customer's historical Monetary spend was the dominant predictive factor (77.2% importance), heavily outweighing both Recency (13.5%) and Frequency (9.3%).

In [22]:
import pandas as pd
from google.colab import userdata
file_path = userdata.get('customer_segmentation')
raw = pd.read_csv(file_path)

In [ ]:
raw.head()

,index,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice,R_Score,F_Score,M_Score,Cluster
0,0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,1,4,5,3
1,1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,1,4,5,3
2,2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,1,4,5,3
3,3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,1,4,5,3
4,4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,1,4,5,3


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

# Assuming your data is loaded into a variable named 'raw'
# raw = pd.read_csv('your_dataset.csv')

# ==========================================
# PHASE 1: DATA CLEANING & WRANGLING
# ==========================================
print("--- Phase 1: Data Cleaning ---")

# 1. Drop transactions missing a CustomerID
df = raw.dropna(subset=['CustomerID']).copy()

# 2. Handle negative values (Returns/Cancellations)
# We strictly want positive quantities and valid prices for clustering and ML
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# 3. Feature engineer TotalSpend
df['TotalSpend'] = df['Quantity'] * df['UnitPrice']

# Ensure InvoiceDate is a proper datetime object
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(f"Initial raw rows: {len(raw)} | Cleaned rows: {len(df)}\n")

# ==========================================
# TEMPORAL SPLIT (Required for Phase 4)
# ==========================================
# We split the data BEFORE Phase 2 so our features don't leak future behavior.
max_date = df['InvoiceDate'].max()
cutoff_date = max_date - pd.Timedelta(days=90) # Last 3 months used as the future target

history_df = df[df['InvoiceDate'] <= cutoff_date]
future_df = df[df['InvoiceDate'] > cutoff_date]

# ==========================================
# PHASE 2: BUILDING THE RFM FRAMEWORK
# ==========================================
print("--- Phase 2: RFM Calculation ---")
snapshot_date = history_df['InvoiceDate'].max() + pd.Timedelta(days=1)

# Group historical data by CustomerID
rfm = history_df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency
    'InvoiceNo': 'nunique',                                  # Frequency
    'TotalSpend': 'sum'                                      # Monetary
}).rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'TotalSpend': 'Monetary'
})

print(rfm.head(), "\n")

# ==========================================
# PHASE 3: BEHAVIORAL CLUSTERING (K-Means)
# ==========================================
print("--- Phase 3: Behavioral Clustering ---")

# 1. Normalize the RFM metrics
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

# 2. Mathematical Proof for Cluster count (Elbow & Silhouette)
# (In a Jupyter Notebook, you would visualize this. Here, we calculate and print.)
inertias = []
silhouette_scores = []
k_range = range(2, 8)

for k in k_range:
    kmeans_test = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans_test.fit_predict(rfm_scaled)
    inertias.append(kmeans_test.inertia_)
    silhouette_scores.append(silhouette_score(rfm_scaled, cluster_labels))

best_k = k_range[np.argmax(silhouette_scores)]
print(f"Optimal number of clusters based on highest Silhouette Score: {best_k}")

# 3. Fit Final K-Means Model
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

# 4. Conceptually Label Clusters based on their Monetary average
# Sort clusters by how much they spend to assign logical business names
cluster_means = rfm.groupby('Cluster')['Monetary'].mean().sort_values()

# Define business labels (from lowest value to highest value)
business_labels = ['Low-Value Shoppers', 'At-Risk / Sleepers', 'Potential Loyalists', 'Champions']

# Create a mapping dictionary and apply it to the dataframe
cluster_map = {cluster_id: label for cluster_id, label in zip(cluster_means.index, business_labels[:best_k])}
rfm['Segment'] = rfm['Cluster'].map(cluster_map)

print(rfm['Segment'].value_counts(), "\n")

# ==========================================
# PHASE 4: CLV FORECASTING (XGBoost)
# ==========================================
print("--- Phase 4: CLV Forecasting ---")

# 1. Calculate Target Variable from the 'future_df' window
target = future_df.groupby('CustomerID')['TotalSpend'].sum().reset_index()
target.rename(columns={'TotalSpend': 'Future_Spend'}, inplace=True)

# 2. Merge historical RFM features with future spend target
dataset = rfm.merge(target, on='CustomerID', how='left')

# If they are in the history but didn't buy in the future, their future spend is 0
dataset['Future_Spend'] = dataset['Future_Spend'].fillna(0)

# 3. Handle Extreme Outliers on the Target
cap_value = dataset['Future_Spend'].quantile(0.99)
dataset = dataset[dataset['Future_Spend'] <= cap_value]

# 4. Prepare Features (X) and Target (y)
# We use continuous variables rather than strict cluster IDs for better predictive variance
X = dataset[['Recency', 'Frequency', 'Monetary']]
y = dataset['Future_Spend']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Build Regression Pipeline using XGBoost (Tweedie objective for zero-inflated data)
model = XGBRegressor(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=4,
    objective='reg:tweedie',
    tweedie_variance_power=1.5, # Ideal for continuous target with many 0s
    random_state=42
)

model.fit(X_train, y_train)

# 6. Make Predictions and Evaluate
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")
print(f"Root Mean Squared Error (RMSE): ${np.sqrt(mse):.2f}")
print(f"R-squared (R2): {r2:.4f}")

# Display Feature Importance
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)
print("\nFeature Importances:\n", importance)

--- Phase 1: Data Cleaning ---
Initial raw rows: 354321 | Cleaned rows: 354321

--- Phase 2: RFM Calculation ---
            Recency  Frequency  Monetary
CustomerID                              
12346.0         235          1  77183.60
12747.0          19          8   2769.40
12748.0           1        132  14107.75
12749.0          40          3   2755.23
12820.0         236          1    170.46 

--- Phase 3: Behavioral Clustering ---
Optimal number of clusters based on highest Silhouette Score: 2
Segment
Low-Value Shoppers    3007
At-Risk / Sleepers      28
Name: count, dtype: int64 

--- Phase 4: CLV Forecasting ---
Training samples: 2403
Testing samples: 601
Root Mean Squared Error (RMSE): $616.12
R-squared (R2): 0.3967

Feature Importances:
      Feature  Importance
2   Monetary    0.772107
0    Recency    0.134715
1  Frequency    0.093178
